# LLMs Overview — Exercises

Companion to the note [LLMs Overview](LLMs%20Overview.md).

Practise the core mechanics of large language models: the causal LM loss and perplexity, causal self-attention and the KV cache, decoding (greedy, beam, temperature, top-k, top-p), scaling laws and back-of-the-envelope compute and memory estimates. All code is NumPy on toy sizes.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from itertools import product
from scipy.special import softmax, log_softmax
rng = np.random.default_rng(0)

# A toy "language model" over tokens A, B, C: start distribution + bigram transition table T[prev, next]
toy_tokens = ['A', 'B', 'C']
toy_start = np.array([0.6, 0.4, 0.0])
toy_T = np.array([[0.28, 0.32, 0.40],
                  [0.90, 0.05, 0.05],
                  [0.34, 0.33, 0.33]])

## Part A · Concepts

### Exercise 1 · Why next-token prediction?
*🧠 Concept · ★☆☆*

The note says LLMs are *decoder-only* [[Transformers]] trained on next-token prediction. Explain (a) why this objective needs
no human labels, (b) what the **causal mask** does and why it is needed during training, and (c) why one forward pass over
a sequence of length $T$ yields $T$ training signals.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Where do the 'labels' $x_t$ come from? What would go wrong if position $t$ could attend to $x_{t+1}$?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) The targets are the text itself shifted by one ($x_t$ given $x_{<t}$), so any raw text is training data: **self-supervised**.
(b) The mask sets attention scores to $-\infty$ for future positions, so the representation at $t$ depends only on $x_{\le t}$.
Without it, the model could simply copy the next token (which it sees in the input) and learn nothing usable at generation time.
(c) Because of the mask, the outputs at all positions are valid predictions of their respective next tokens, so the loss
$-\sum_t \log p(x_t\mid x_{<t})$ uses all $T$ positions in parallel ("teacher forcing"). Generation, in contrast, is sequential.

</details>

### Exercise 2 · Choosing a decoding strategy
*🧠 Concept · ★☆☆*

For each use case pick greedy, beam search, or sampling with temperature / top-p, and justify:
(a) translating a legal sentence, (b) writing a short story, (c) generating code that must pass unit tests (you may sample
several candidates), (d) a chatbot. What failure mode does greedy/beam decoding show in open-ended generation?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Beam search maximises sequence likelihood; the highest-likelihood text is not always the most human-like.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Beam search** (or greedy): there is one "correct" output and we want the most likely one.
(b) **Sampling** with $T\approx 0.8$–$1$ and top-p $\approx 0.9$: diversity and creativity matter.
(c) **Sample $k$ candidates** at moderate temperature and keep the ones that pass tests (pass@k); low temperature if only one try.
(d) Sampling with modest temperature / top-p: natural but not erratic.
Greedy/beam in open-ended generation produce **repetitive, bland loops** ("I don't know. I don't know. …"), because
high-probability continuations reinforce themselves; human text is not the argmax (Holtzman et al. 2020, which introduced top-p).

</details>

### Exercise 3 · In-context learning and contamination
*🧠 Concept · ★★☆*

(a) What is the difference between few-shot **in-context learning** and **fine-tuning**? Do the weights change?
(b) Why can chain-of-thought prompting improve accuracy on GSM8K-style problems?
(c) The note warns "beware contamination". What is it, and how could you detect it for a benchmark?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For (b), think about how much computation a transformer can do per generated token.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) ICL puts examples in the **prompt**; the weights are frozen and the "learning" happens in the forward pass (attention over
the examples). Fine-tuning updates the weights ([[Fine-tuning and Alignment]]). ICL is instant and cheap per task but limited by
context length and less reliable.
(b) Each token gets a fixed amount of computation; writing intermediate steps lets the model spend more tokens (more serial
computation) and condition later steps on earlier results, like a scratchpad.
(c) **Contamination**: benchmark test items (or close paraphrases) were in the pretraining data, so scores measure memorisation.
Detection: n-gram overlap search of the training corpus, testing on freshly written or perturbed versions of the items (a large
drop is suspicious), checking whether the model can complete the test question verbatim, or comparing perplexity on test vs.
similar unseen items.

</details>

### Exercise 4 · Mixture of experts: capacity vs compute
*🧠 Concept · ★★☆*

A model replaces each dense FFN block with 8 expert FFNs of the same size and a router that sends each token to the top-2.
Relative to the dense model: how do (a) total parameters in the FFN blocks, (b) FLOPs per token in the FFN blocks,
(c) GPU memory to hold the weights change? Why is load balancing needed?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Every expert must be stored, but each token only passes through two of them.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) FFN parameters grow **8×** (capacity / knowledge). (b) FLOPs per token grow only **2×** (two experts active), plus a tiny router
cost. (c) Memory grows with *total* parameters, so the weights still need ~8× FFN memory: MoE trades memory for compute.
Without a **load-balancing** auxiliary loss, the router tends to collapse onto a few experts (rich-get-richer); the others are
undertrained and capacity is wasted, and in distributed serving some devices overflow while others idle.

</details>

## Part B · By hand

### Exercise 5 · Causal LM loss and perplexity
*✍️ By hand · ★☆☆*

A model assigns the three target tokens of a sequence the probabilities $0.5$, $0.25$ and $0.8$.
Compute the summed loss $-\sum_t\log p(x_t\mid x_{<t})$ (natural log), the mean loss per token, and the perplexity $\exp(\text{mean loss})$.

In [ ]:
loss_sum = None
loss_mean = None
ppl = None

_p = np.array([0.5, 0.25, 0.8])
check('summed loss', loss_sum, -np.log(_p).sum())
check('mean loss', loss_mean, -np.log(_p).mean())
check('perplexity', ppl, np.exp(-np.log(_p).mean()))

<details>
<summary><b>💡 Hint</b></summary>

The product of the probabilities is $0.1$, so the summed loss is $\ln 10$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Summed loss $= -\ln(0.5\cdot0.25\cdot0.8) = \ln 10 \approx 2.303$; mean $\approx 0.768$; perplexity
$= e^{0.768} = 10^{1/3} \approx 2.154$. The loss is what the optimiser minimises; perplexity is the more interpretable
"effective branching factor".

```python
loss_sum = np.log(10)
loss_mean = np.log(10) / 3
ppl = 10 ** (1 / 3)
```

</details>

### Exercise 6 · Temperature
*✍️ By hand · ★☆☆*

Logits $z = (2, 1, 0)$. Sampling uses $p_i = \text{softmax}(z/T)_i$. Compute the probability of the first token for
$T = 1$, $T = 0.5$ and $T = 2$. What happens as $T\to 0$ and $T\to\infty$?

In [ ]:
p_T1 = None
p_T05 = None
p_T2 = None

_z = np.array([2., 1., 0.])
for _n, _v, _t in [('T=1', p_T1, 1), ('T=0.5', p_T05, 0.5), ('T=2', p_T2, 2)]:
    check(_n, _v, softmax(_z / _t)[0], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

For $T=0.5$ the scaled logits are $(4,2,0)$: $p_1 = e^4/(e^4+e^2+1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$T=1$: $e^2/(e^2+e+1) = 7.389/11.107 \approx 0.665$. $T=0.5$: $54.60/62.99 \approx 0.867$. $T=2$: $e/(e + e^{0.5} + 1) \approx 0.506$.
$T\to0$ puts all mass on the argmax (greedy decoding); $T\to\infty$ gives the uniform distribution. Temperature does not change the
*ranking* of tokens, only how peaked the distribution is.

```python
p_T1 = np.exp(2) / (np.exp(2) + np.e + 1)
p_T05 = np.exp(4) / (np.exp(4) + np.exp(2) + 1)
p_T2 = np.e / (np.e + np.exp(0.5) + 1)
```

</details>

### Exercise 7 · Top-k and top-p (nucleus) by hand
*✍️ By hand · ★☆☆*

Next-token probabilities (sorted): $(0.5, 0.2, 0.15, 0.1, 0.05)$.
(a) With **top-k**, $k=2$: which tokens remain and what is the renormalised probability of the first?
(b) With **top-p**, $p=0.8$ (keep the smallest prefix whose cumulative mass is $\ge p$): how many tokens remain, and what is the renormalised probability of the first?

In [ ]:
p_first_topk2 = None
n_kept_topp = None
p_first_topp = None

_p = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
_n = int(np.searchsorted(np.cumsum(_p), 0.8 - 1e-12) + 1)
check('top-k first', p_first_topk2, _p[0] / _p[:2].sum())
check('top-p kept', n_kept_topp, _n)
check('top-p first', p_first_topp, _p[0] / _p[:_n].sum())

<details>
<summary><b>💡 Hint</b></summary>

Cumulative sums: 0.5, 0.7, 0.85, 0.95, 1.0.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Keep $\{0.5, 0.2\}$: $0.5/0.7 \approx 0.714$.
(b) The cumulative mass first reaches $0.8$ at the third token ($0.85$), so 3 tokens remain and the first gets $0.5/0.85 \approx 0.588$.
Top-p adapts the number of candidates to the model's confidence: few when it is sure, many when the distribution is flat,
while top-k always keeps exactly $k$.

```python
p_first_topk2 = 0.5 / 0.7
n_kept_topp = 3
p_first_topp = 0.5 / 0.85
```

</details>

### Exercise 8 · Chinchilla-optimal training
*✍️ By hand · ★★☆*

Training compute is approximately $C \approx 6ND$ FLOPs ($N$ parameters, $D$ tokens), and the note's Chinchilla rule is $D\approx 20N$.
(a) How many tokens should a 7B-parameter model see, and how many FLOPs does that cost?
(b) With a budget of $C = 10^{24}$ FLOPs, what is the compute-optimal $N$?

In [ ]:
D_7b = None      # tokens
C_7b = None      # FLOPs
N_opt = None     # parameters for C = 1e24

check('tokens', D_7b, 20 * 7e9)
check('FLOPs', C_7b, 6 * 7e9 * 20 * 7e9)
check('N_opt', N_opt, np.sqrt(1e24 / 120), tol=1e8)

<details>
<summary><b>💡 Hint</b></summary>

Substitute $D = 20N$ into $C = 6ND$: $C = 120 N^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $D = 1.4\times10^{11}$ tokens (140B), $C = 6\cdot 7\times10^9\cdot1.4\times10^{11} = 5.88\times10^{21}$ FLOPs.
(b) $N = \sqrt{C/120} = \sqrt{8.33\times10^{21}} \approx 9.1\times10^{10}$, i.e. about 91B parameters trained on 1.8T tokens.
In practice models like Llama are trained *far beyond* 20 tokens/parameter, because a smaller model is cheaper at inference time:
Chinchilla optimises training compute only.

```python
D_7b = 1.4e11
C_7b = 6 * 7e9 * 1.4e11
N_opt = np.sqrt(1e24 / 120)
```

</details>

### Exercise 9 · Counting transformer parameters
*✍️ By hand · ★★☆*

Each decoder block has attention projections $W_Q, W_K, W_V, W_O \in\mathbb R^{d\times d}$ and an FFN $d\to4d\to d$
(ignore biases and LayerNorms). Show that a block has about $12d^2$ parameters, then compute the non-embedding parameters of
GPT-2 small ($L=12$, $d=768$) and the token-embedding parameters (vocabulary 50 257).

In [ ]:
params_blocks = None
params_embed = None

check('block params', params_blocks, 12 * (4 * 768**2 + 2 * 4 * 768**2) / 1)
check('embedding params', params_embed, 50257 * 768)

<details>
<summary><b>💡 Hint</b></summary>

Attention: $4d^2$. FFN: $d\cdot 4d + 4d\cdot d = 8d^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Per block $4d^2 + 8d^2 = 12d^2$. For GPT-2 small: $12\cdot 12\cdot 768^2 = 84{,}934{,}656$ ($\approx 85$M);
embeddings $50{,}257\cdot768 = 38{,}597{,}376$ ($\approx 39$M). Total $\approx 124$M with position embeddings, matching the
published "124M". For large models the $12Ld^2$ term dominates.

```python
params_blocks = 12 * 12 * 768**2
params_embed = 50257 * 768
```

</details>

### Exercise 10 · KV-cache and weight memory
*✍️ By hand · ★★☆*

A 7B model has $L=32$ layers, $d = 4096$ (32 heads of size 128) and is served in fp16 (2 bytes per number).
(a) How many bytes do the weights need, and how many with 4-bit quantisation?
(b) The KV cache stores keys **and** values for every layer and position. How many bytes for one sequence of 4096 tokens?

In [ ]:
weights_fp16 = None
weights_int4 = None
kv_bytes = None

check('fp16 weights', weights_fp16, 7e9 * 2)
check('int4 weights', weights_int4, 7e9 * 0.5)
check('KV cache', kv_bytes, 2 * 32 * 4096 * 4096 * 2)

<details>
<summary><b>💡 Hint</b></summary>

KV cache $= 2\,(K,V) \times L \times T \times d \times \text{bytes}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $14$ GB in fp16, $3.5$ GB in 4-bit (plus a little for scales).
(b) $2\cdot32\cdot4096\cdot4096\cdot2 = 2^{31}$ bytes $= 2$ GiB **per sequence**. With a batch of 16 long conversations the cache
exceeds the weights, which is why grouped-query attention (fewer K/V heads), paged attention and KV quantisation exist.

```python
weights_fp16 = 14e9
weights_int4 = 3.5e9
kv_bytes = 2 * 32 * 4096 * 4096 * 2
```

</details>

## Part C · Code from scratch

### Exercise 11 · Stable cross-entropy from logits
*💻 Code · ★☆☆*

Implement `lm_loss(logits, targets)` returning the mean next-token cross-entropy for logits of shape `(T, V)` and integer
targets of shape `(T,)`. Use the log-sum-exp trick so that logits around 1000 do not overflow.

In [ ]:
def lm_loss(logits, targets):
    # TODO: mean of -log softmax(logits)[t, targets[t]]
    return None

lg = rng.normal(size=(6, 10)) * 3 + 1000
tg = rng.integers(0, 10, size=6)
my_loss = lm_loss(lg, tg)

check('loss', my_loss, -log_softmax(lg, axis=1)[np.arange(6), tg].mean())

<details>
<summary><b>💡 Hint</b></summary>

$\log\text{softmax}(z)_i = z_i - m - \log\sum_j e^{z_j - m}$ with $m = \max_j z_j$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Subtracting the row maximum changes nothing mathematically but keeps every exponent $\le 0$. Frameworks fuse softmax and
cross-entropy for exactly this reason (`F.cross_entropy` takes logits, not probabilities).

```python
def lm_loss(logits, targets):
    m = logits.max(axis=1, keepdims=True)
    logp = logits - m - np.log(np.exp(logits - m).sum(axis=1, keepdims=True))
    return -logp[np.arange(len(targets)), targets].mean()

lg = rng.normal(size=(6, 10)) * 3 + 1000
tg = rng.integers(0, 10, size=6)
my_loss = lm_loss(lg, tg)
```

</details>

### Exercise 12 · Causal self-attention
*💻 Code · ★★☆*

Implement single-head causal attention `causal_attention(Q, K, V)` = $\text{softmax}\big(\frac{QK^\top}{\sqrt{d_k}} + M\big)V$
where $M_{ij} = -\infty$ for $j > i$. Fully vectorised, no loops.

In [ ]:
def causal_attention(Q, K, V):
    # TODO
    return None

Tq, dk = 5, 4
Qa, Ka, Va = rng.normal(size=(3, Tq, dk))
att_out = causal_attention(Qa, Ka, Va)

_ref = np.zeros((Tq, dk))
for _i in range(Tq):                      # brute force: row i attends to positions 0..i only
    _s = Qa[_i] @ Ka[:_i + 1].T / np.sqrt(dk)
    _ref[_i] = softmax(_s) @ Va[:_i + 1]
check('causal attention', att_out, _ref)

<details>
<summary><b>💡 Hint</b></summary>

`mask = np.triu(np.ones((T, T), bool), k=1)`; `scores[mask] = -np.inf`; then a row-wise stable softmax.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Row $i$ of the output is a convex combination of $V_0,\dots,V_i$ only. Row 0 equals $V_0$ exactly (it can only attend to itself).
Changing $V_4$ changes only the last output row: that is the causal property that makes teacher forcing valid.

```python
def causal_attention(Q, K, V):
    T = Q.shape[0]
    S = Q @ K.T / np.sqrt(K.shape[1])
    S = np.where(np.triu(np.ones((T, T), bool), k=1), -np.inf, S)
    S = S - S.max(axis=1, keepdims=True)
    A = np.exp(S)
    A /= A.sum(axis=1, keepdims=True)
    return A @ V

Tq, dk = 5, 4
Qa, Ka, Va = rng.normal(size=(3, Tq, dk))
att_out = causal_attention(Qa, Ka, Va)
```

</details>

### Exercise 13 · Incremental decoding with a KV cache
*💻 Code · ★★★*

During generation, token $t$ only needs its own query, but the keys/values of all previous tokens. Implement
`decode_with_cache(X, Wq, Wk, Wv)` that processes the rows of `X` **one at a time**: compute $q_t, k_t, v_t$, append $k_t, v_t$
to the cache, and output $\text{softmax}(q_t K_{\le t}^\top/\sqrt{d})\,V_{\le t}$. Return the stacked outputs; they must equal full
causal attention on $Q = XW_q$, $K = XW_k$, $V = XW_v$. How much work does step $t$ do with and without the cache?

In [ ]:
def decode_with_cache(X, Wq, Wk, Wv):
    # TODO
    return None

Xc = rng.normal(size=(6, 8))
Wq_, Wk_, Wv_ = rng.normal(size=(3, 8, 4))
cached_out = decode_with_cache(Xc, Wq_, Wk_, Wv_)

_Q, _K, _V = Xc @ Wq_, Xc @ Wk_, Xc @ Wv_
_full = np.stack([softmax(_Q[i] @ _K[:i + 1].T / 2.0) @ _V[:i + 1] for i in range(6)])
check('cache == full causal attention', cached_out, _full)

<details>
<summary><b>💡 Hint</b></summary>

Keep Python lists `Ks, Vs`; at step $t$ stack them into arrays of shape $(t+1, d)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the cache, step $t$ computes one new $q,k,v$ (cost $O(d^2)$) and one attention row ($O(t\,d)$). Without it, you would recompute
$K, V$ for all $t$ previous tokens at every step ($O(t\,d^2)$ per step, $O(T^2 d^2)$ total). The price is the memory computed in
the KV-cache exercise above.

```python
def decode_with_cache(X, Wq, Wk, Wv):
    Ks, Vs, outs = [], [], []
    for x in X:
        q, k, v = x @ Wq, x @ Wk, x @ Wv
        Ks.append(k); Vs.append(v)
        K, V = np.array(Ks), np.array(Vs)
        s = q @ K.T / np.sqrt(len(q))
        a = np.exp(s - s.max()); a /= a.sum()
        outs.append(a @ V)
    return np.array(outs)

Xc = rng.normal(size=(6, 8))
Wq_, Wk_, Wv_ = rng.normal(size=(3, 8, 4))
cached_out = decode_with_cache(Xc, Wq_, Wk_, Wv_)
```

</details>

### Exercise 14 · Temperature, top-k and top-p in one function
*💻 Code · ★★☆*

Implement `filter_probs(logits, temperature=1.0, top_k=None, top_p=None)` that returns the **renormalised** sampling
distribution: apply temperature, then keep the top-$k$ tokens, then keep the smallest set whose mass is $\ge$ `top_p`
(computed on the distribution after top-$k$). Then draw 10 000 samples with `rng.choice` and compare the empirical frequencies.

In [ ]:
def filter_probs(logits, temperature=1.0, top_k=None, top_p=None):
    # TODO
    return None

probs5 = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
logits5 = np.log(probs5)
fp = filter_probs(logits5, top_p=0.8)

check('temperature only', filter_probs(np.array([2., 1., 0.]), temperature=0.5), softmax(np.array([4., 2., 0.])))
check('top-k = 2', filter_probs(logits5, top_k=2), np.r_[0.5, 0.2, 0, 0, 0] / 0.7)
check('top-p = 0.8', fp, np.r_[0.5, 0.2, 0.15, 0, 0] / 0.85)

<details>
<summary><b>💡 Hint 1</b></summary>

Sort indices by descending probability with `np.argsort(-p)`; zero out everything past position $k$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For top-p: `n = np.searchsorted(np.cumsum(sorted_p), top_p) + 1` tokens are kept (careful with ties at exactly `top_p`).

</details>

<details>
<summary><b>✅ Solution</b></summary>

The order of operations matters and libraries differ; the common convention (Hugging Face) is temperature → top-k → top-p.
Empirical frequencies from 10 000 samples match the filtered distribution to about $\pm0.01$.

```python
def filter_probs(logits, temperature=1.0, top_k=None, top_p=None):
    z = np.asarray(logits, float) / temperature
    p = np.exp(z - z.max()); p /= p.sum()
    order = np.argsort(-p, kind='stable')
    keep = np.zeros(len(p), bool)
    keep[order[:top_k] if top_k else order] = True
    p = np.where(keep, p, 0); p /= p.sum()
    if top_p is not None:
        cs = np.cumsum(p[order])
        n = int(np.searchsorted(cs, top_p - 1e-12)) + 1
        mask = np.zeros(len(p), bool); mask[order[:n]] = True
        p = np.where(mask, p, 0); p /= p.sum()
    return p

probs5 = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
logits5 = np.log(probs5)
fp = filter_probs(logits5, top_p=0.8)
draws = rng.choice(5, size=10_000, p=fp)
print(np.round(fp, 3), np.bincount(draws, minlength=5) / 10_000)
```

</details>

### Exercise 15 · Greedy vs beam search
*💻 Code · ★★★*

The setup defines a toy LM over tokens A, B, C: `toy_start[i]` = $P(x_1=i)$ and `toy_T[i, j]` = $P(x_{t+1}=j\mid x_t=i)$.
For sequences of length 3:
1. implement `greedy(start, T, length)` returning `(sequence, probability)`;
2. implement `beam_search(start, T, length, width)` (work with log-probabilities);
3. find the most probable sequence by brute force over all $3^3$ sequences.

Store the probabilities `p_greedy`, `p_beam2` (width 2), `p_beam3` (width 3) and the brute-force sequence `best_seq` (e.g. `'ABC'`).

In [ ]:
def greedy(start, T, length):
    # TODO
    return None

def beam_search(start, T, length, width):
    # TODO: return (sequence as list of ints, probability)
    return None

p_greedy = None
p_beam2 = None
p_beam3 = None
best_seq = None

_all = {''.join(toy_tokens[i] for i in s): toy_start[s[0]] * toy_T[s[0], s[1]] * toy_T[s[1], s[2]] for s in product(range(3), repeat=3)}
_best = max(_all, key=_all.get)
check('brute-force best', best_seq, _best)
check('beam (width 3) finds it', p_beam3, _all[_best])
check('greedy < beam-2 < optimum', None if p_greedy is None else p_greedy < p_beam2 < _all[_best], True)

<details>
<summary><b>💡 Hint 1</b></summary>

Greedy: A (0.6) is the best first token, but after A the next tokens are spread out, while B is almost always followed by A.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Beam: keep the `width` best partial sequences by total log-probability after each step, expanding each by all tokens.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Greedy picks A (0.6) → C (0.4) → A (0.34): probability $0.0816$.
Beam search with width 2 keeps B (0.4) alive; after step 2 its beams are BA ($0.36$) and AC ($0.24$), and it returns
**BAC** with $0.4\cdot0.9\cdot0.4 = 0.144$. But it pruned AB ($0.192$), and the brute-force optimum is
**ABA** $= 0.6\cdot0.32\cdot0.9 = 0.1728$, which width 3 finds.
Lessons: greedy is myopic, beam search is a better but still approximate search whose quality grows with the width, and (as
discussed above) the most likely sequence is not always the best text anyway.

```python
def greedy(start, T, length):
    seq = [int(np.argmax(start))]; p = start[seq[0]]
    for _ in range(length - 1):
        nxt = int(np.argmax(T[seq[-1]])); p *= T[seq[-1], nxt]; seq.append(nxt)
    return seq, p

def beam_search(start, T, length, width):
    with np.errstate(divide='ignore'):
        beams = [([i], np.log(start[i])) for i in range(len(start))]
        beams = sorted(beams, key=lambda b: -b[1])[:width]
        for _ in range(length - 1):
            cand = [(s + [j], lp + np.log(T[s[-1], j])) for s, lp in beams for j in range(T.shape[1])]
            beams = sorted(cand, key=lambda b: -b[1])[:width]
    return beams[0][0], float(np.exp(beams[0][1]))

g_seq, p_greedy = greedy(toy_start, toy_T, 3)
b_seq, p_beam2 = beam_search(toy_start, toy_T, 3, 2)
_, p_beam3 = beam_search(toy_start, toy_T, 3, 3)
scores = {''.join(toy_tokens[i] for i in s): toy_start[s[0]] * toy_T[s[0], s[1]] * toy_T[s[1], s[2]] for s in product(range(3), repeat=3)}
best_seq = max(scores, key=scores.get)
print([toy_tokens[i] for i in g_seq], p_greedy, [toy_tokens[i] for i in b_seq], p_beam2, p_beam3, best_seq)
```

</details>

### Exercise 16 · Fitting a scaling law
*💻 Code · ★★☆*

Kaplan et al. found $L(N) = (N_c/N)^{\alpha_N}$. The cell below simulates noisy losses for 9 model sizes with
$N_c = 8.8\times10^{13}$ and $\alpha_N = 0.076$. Estimate $\alpha_N$ and $N_c$ by linear regression in log–log space, then
predict the loss of a $10^{12}$-parameter model.

In [ ]:
N_sizes = np.logspace(6, 10, 9)
losses = (8.8e13 / N_sizes) ** 0.076 * np.exp(0.005 * rng.normal(size=9))
alpha_hat = None
Nc_hat = None
loss_1T = None

check('alpha', alpha_hat, 0.076, tol=0.005)
check('predicted loss at 1e12', loss_1T, (8.8e13 / 1e12) ** 0.076, tol=0.05)

<details>
<summary><b>💡 Hint</b></summary>

$\log L = \alpha\log N_c - \alpha\log N$: a straight line with slope $-\alpha$ and intercept $\alpha\log N_c$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

`np.polyfit(np.log(N), np.log(L), 1)` gives slope $\approx -0.076$ and intercept $\alpha\log N_c$, so
$N_c = \exp(\text{intercept}/\alpha)$. The prediction for $10^{12}$ parameters is $\approx (88)^{0.076} \approx 1.40$. Power laws are straight
lines on log–log axes, which is how such extrapolations (and Chinchilla's compute-optimal allocation) are made. Extrapolating
two orders of magnitude is risky: small slope errors blow up.

```python
N_sizes = np.logspace(6, 10, 9)
losses = (8.8e13 / N_sizes) ** 0.076 * np.exp(0.005 * rng.normal(size=9))
slope, intercept = np.polyfit(np.log(N_sizes), np.log(losses), 1)
alpha_hat = -slope
Nc_hat = np.exp(intercept / alpha_hat)
loss_1T = (Nc_hat / 1e12) ** alpha_hat
print(alpha_hat, f"{Nc_hat:.2e}", loss_1T)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [LLMs Overview](LLMs%20Overview.md).*